# Build the helper tensors for a sequence batch

Before a sequence model runs, the training script builds a handful of small tensors: position ids, a padding mask, a causal mask, a time grid. Each one is a single PyTorch factory call if you know which one to reach for.

Write `make_helpers(lengths, T, t0, t1)`:

- `lengths`: int64 tensor `(B,)`, the number of real tokens in each sequence ($0 \le$ `lengths[b]` $\le T$).
- `T`: int, the padded sequence length ($T \ge 1$).
- `t0`, `t1`: floats, the ends of a time grid.

Return a tuple of five tensors:

| # | Name | Shape | dtype | Contents |
|---|---|---|---|---|
| 1 | `pos` | `(T,)` | int64 | `0, 1, …, T-1` |
| 2 | `pad_mask` | `(B, T)` | bool | `True` where position `t < lengths[b]` |
| 3 | `causal` | `(T, T)` | bool | `True` where key `j <= ` query `i` (lower triangle, diagonal included) |
| 4 | `grid` | `(T,)` | float64 | `T` evenly spaced points from `t0` to `t1`, both ends included (for `T = 1` it is `[t0]`) |
| 5 | `bias` | `(B, T)` | float64 | `0.0` at real positions, `-inf` at padded ones |

Use factory functions and broadcasting, no Python loops. The grader compares values, so `True`/`1` both pass, but get the dtypes right anyway: they matter the moment these tensors meet a model.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def make_helpers(lengths, T, t0, t1):
    """Return (pos, pad_mask, causal, grid, bias). See the table for shapes and dtypes."""
    pos = torch.zeros(T)
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [2, 3], "dtype": "int64"}, 3, 0.0, 1.0], "e": {"__tup__": [{"__nd__": [0, 1, 2], "dtype": "int64"}, {"__nd__": [[true, true, false], [true, true, true]], "dtype": "bool"}, {"__nd__": [[true, false, false], [true, true, false], [true, true, true]], "dtype": "bool"}, {"__nd__": [0.0, 0.5, 1.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, {"__f__": "-inf"}], [0.0, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [1], "dtype": "int64"}, 4, -1.0, 1.0], "e": {"__tup__": [{"__nd__": [0, 1, 2, 3], "dtype": "int64"}, {"__nd__": [[true, false, false, false]], "dtype": "bool"}, {"__nd__": [[true, false, false, false], [true, true, false, false], [true, true, true, false], [true, true, true, true]], "dtype": "bool"}, {"__nd__": [-1.0, -0.33333333333333337, 0.33333333333333337, 1.0], "dtype": "float64"}, {"__nd__": [[0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float64"}]}}, {"a": [{"__nd__": [0, 5, 2], "dtype": "int64"}, 5, 0.0, 10.0], "e": {"__tup__": [{"__nd__": [0, 1, 2, 3, 4], "dtype": "int64"}, {"__nd__": [[false, false, false, false, false], [true, true, true, true, true], [true, true, false, false, false]], "dtype": "bool"}, {"__nd__": [[true, false, false, false, false], [true, true, false, false, false], [true, true, true, false, false], [true, true, true, true, false], [true, true, true, true, true]], "dtype": "bool"}, {"__nd__": [0.0, 2.5, 5.0, 7.5, 10.0], "dtype": "float64"}, {"__nd__": [[{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float64"}]}}, {"a": [{"__nd__": [1, 0], "dtype": "int64"}, 1, 3.0, 7.0], "e": {"__tup__": [{"__nd__": [0], "dtype": "int64"}, {"__nd__": [[true], [false]], "dtype": "bool"}, {"__nd__": [[true]], "dtype": "bool"}, {"__nd__": [3.0], "dtype": "float64"}, {"__nd__": [[0.0], [{"__f__": "-inf"}]], "dtype": "float64"}]}}, {"a": [{"__nd__": [2, 0], "dtype": "int64"}, 2, 0.0, 0.0], "e": {"__tup__": [{"__nd__": [0, 1], "dtype": "int64"}, {"__nd__": [[true, true], [false, false]], "dtype": "bool"}, {"__nd__": [[true, false], [true, true]], "dtype": "bool"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [{"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float64"}]}}, {"a": [{"__nd__": [6, 1, 4, 6], "dtype": "int64"}, 6, 2.5, -2.5], "e": {"__tup__": [{"__nd__": [0, 1, 2, 3, 4, 5], "dtype": "int64"}, {"__nd__": [[true, true, true, true, true, true], [true, false, false, false, false, false], [true, true, true, true, false, false], [true, true, true, true, true, true]], "dtype": "bool"}, {"__nd__": [[true, false, false, false, false, false], [true, true, false, false, false, false], [true, true, true, false, false, false], [true, true, true, true, false, false], [true, true, true, true, true, false], [true, true, true, true, true, true]], "dtype": "bool"}, {"__nd__": [2.5, 1.5, 0.5, -0.5, -1.5, -2.5], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [0.0, 0.0, 0.0, 0.0, {"__f__": "-inf"}, {"__f__": "-inf"}], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [7, 3, 0, 5, 1], "dtype": "int64"}, 7, 0.0, 0.001], "e": {"__tup__": [{"__nd__": [0, 1, 2, 3, 4, 5, 6], "dtype": "int64"}, {"__nd__": [[true, true, true, true, true, true, true], [true, true, true, false, false, false, false], [false, false, false, false, false, false, false], [true, true, true, true, true, false, false], [true, false, false, false, false, false, false]], "dtype": "bool"}, {"__nd__": [[true, false, false, false, false, false, false], [true, true, false, false, false, false, false], [true, true, true, false, false, false, false], [true, true, true, true, false, false, false], [true, true, true, true, true, false, false], [true, true, true, true, true, true, false], [true, true, true, true, true, true, true]], "dtype": "bool"}, {"__nd__": [0.0, 0.00016666666666666666, 0.0003333333333333333, 0.0005, 0.0006666666666666668, 0.0008333333333333334, 0.001], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [0.0, 0.0, 0.0, 0.0, 0.0, {"__f__": "-inf"}, {"__f__": "-inf"}], [0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(make_helpers, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")